# 04 - Regression and Academic-Risk Classification

This notebook compares regression models for `Final_CGPA` and classifiers for a separately defined, provisional academic-risk label. Models are compared on training-only cross-validation, then selected pipelines are evaluated on held-out data. Results describe this dataset and do not guarantee performance for a different cohort. Predictive associations are not causes.

## 1. Setup and data checks

A fixed seed makes splits and randomized estimators repeatable. The path is relative to the project root. `Student_ID` is kept for record identity but is excluded from model inputs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
    make_scorer, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
RISK_THRESHOLD = 2.0  # Provisional project assumption on a 0 to 4 CGPA scale.
TARGET_COLUMN = "Final_CGPA"
IDENTIFIER_COLUMN = "Student_ID"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw" / "Student_data.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Student_data.csv"
assert DATA_PATH.is_file(), "Expected data/raw/Student_data.csv relative to the project root."
data = pd.read_csv(DATA_PATH)
assert TARGET_COLUMN in data and IDENTIFIER_COLUMN in data
assert pd.api.types.is_numeric_dtype(data[TARGET_COLUMN]) and data[TARGET_COLUMN].notna().all()
assert not data.duplicated().any(), "Unexpected duplicate rows found."
feature_columns = [c for c in data.columns if c not in {TARGET_COLUMN, IDENTIFIER_COLUMN}]
X = data[feature_columns].copy()
y_regression = data[TARGET_COLUMN].astype(float).copy()
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Dataset: {data.shape[0]:,} rows by {data.shape[1]} columns")
print(f"Candidate inputs ({len(feature_columns)}): {feature_columns}")
print(f"Numeric: {numeric_features}")
print(f"Categorical: {categorical_features}")
assert TARGET_COLUMN not in X and IDENTIFIER_COLUMN not in X and len(feature_columns) == 8

## 2. Leakage-safe preprocessing

This matches Module 2: median imputation and scaling for numeric columns, most-frequent imputation and one-hot encoding for categorical columns. Each estimator pipeline contains its own preprocessor. Cross-validation therefore learns imputation values, scales, and categories inside each training fold, without seeing that fold's validation rows. Neither `Final_CGPA` nor a target-derived label is an input.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())])
    categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                            ("one_hot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("numeric", numeric, numeric_features),
                              ("categorical", categorical, categorical_features)],
                             remainder="drop")

def make_pipeline(estimator):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])

## 3. Regression: predict Final_CGPA

We use the same reproducible 80/20 split as Module 2. Five-fold shuffled cross-validation compares candidates using training rows only. MAE and RMSE are measured in CGPA points (lower is better); R2 compares performance with a constant-mean reference (higher is generally better and can be negative). The selected model is the one with lowest mean cross-validation MAE. Test results are not used to choose it.

The dummy regressor predicts a training-fold mean and provides a simple baseline.

In [ ]:
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X, y_regression, test_size=TEST_SIZE, random_state=RANDOM_STATE)
reg_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
reg_estimators = {
    "Dummy mean baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}
reg_scoring = {"mae": "neg_mean_absolute_error",
               "rmse": "neg_root_mean_squared_error", "r2": "r2"}
reg_pipelines, reg_rows = {}, []
for name, estimator in reg_estimators.items():
    pipe = make_pipeline(estimator)
    reg_pipelines[name] = pipe
    scores = cross_validate(pipe, X_reg_train, y_reg_train, cv=reg_cv,
                            scoring=reg_scoring, n_jobs=1, error_score="raise")
    reg_rows.append({"model": name,
        "CV MAE mean": -scores["test_mae"].mean(), "CV MAE std": scores["test_mae"].std(),
        "CV RMSE mean": -scores["test_rmse"].mean(), "CV RMSE std": scores["test_rmse"].std(),
        "CV R2 mean": scores["test_r2"].mean(), "CV R2 std": scores["test_r2"].std()})
reg_cv_results = pd.DataFrame(reg_rows).sort_values("CV MAE mean")
display(reg_cv_results.round(4))
selected_regression_name = reg_cv_results.iloc[0]["model"]
selected_regression = reg_pipelines[selected_regression_name]
print(f"Selected using training CV MAE: {selected_regression_name}")
assert len(X_reg_train) == 4000 and len(X_reg_test) == 1000
assert TARGET_COLUMN not in X_reg_train and IDENTIFIER_COLUMN not in X_reg_train

### Held-out regression evaluation

Now that training-fold results have selected the model, fit it using all regression training rows and score the held-out test rows. The dummy baseline is also shown as a reference. Predictions remain unclipped; the count outside the observed target range is reported rather than silently changed.

In [ ]:
selected_regression.fit(X_reg_train, y_reg_train)
selected_predictions = selected_regression.predict(X_reg_test)
dummy_test = reg_pipelines["Dummy mean baseline"].fit(X_reg_train, y_reg_train)
dummy_predictions = dummy_test.predict(X_reg_test)

def regression_metrics(actual, predicted):
    return {"MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "R2": r2_score(actual, predicted)}

reg_test_results = pd.DataFrame([
    {"model": "Dummy mean baseline", **regression_metrics(y_reg_test, dummy_predictions)},
    {"model": selected_regression_name, **regression_metrics(y_reg_test, selected_predictions)},
])
display(reg_test_results.round(4))
y_min, y_max = float(y_regression.min()), float(y_regression.max())
outside = (selected_predictions < y_min) | (selected_predictions > y_max)
print(f"Observed target range: {y_min:.2f} to {y_max:.2f}")
print(f"Unclipped selected-model prediction range: {selected_predictions.min():.3f} to {selected_predictions.max():.3f}")
print(f"Predictions outside observed range: {int(outside.sum())} / {len(outside)}")
assert len(selected_predictions) == len(y_reg_test) and np.isfinite(selected_predictions).all()

## 4. Academic-risk classification: definition and split

The configurable default `RISK_THRESHOLD = 2.0` labels a record at risk when `Final_CGPA < 2.0`; exactly 2.0 is not at risk. This is a **provisional project assumption** on the dataset's 0 to 4 scale, not an official or verified university policy. Replace it with a documented institutional rule if one is provided. This target-label threshold is distinct from the classifier's default 0.5 probability cutoff.

False negatives are below-threshold records the classifier misses; false positives are above-threshold records it flags. Prioritizing recall may identify more potential risk cases, while raising false alarms. A score is a prompt for human review, not a diagnosis or consequential decision by itself.

In [ ]:
# The outcome-derived label is y only; it and Final_CGPA are absent from X.
y_risk = (y_regression < RISK_THRESHOLD).astype(int).rename("At_Risk")
class_balance = y_risk.value_counts().reindex([0, 1], fill_value=0)
class_balance.index = ["Not at risk (0)", "At risk (1)"]
class_balance = class_balance.rename("records").to_frame()
class_balance["percent"] = 100 * class_balance["records"] / len(y_risk)
print(f"Assumed label rule: Final_CGPA < {RISK_THRESHOLD:.2f}")
display(class_balance.round(2))
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X, y_risk, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_risk)
print("Training class percentages:")
display((100*y_cls_train.value_counts(normalize=True).reindex([0,1], fill_value=0)).rename("percent").to_frame())
print("Test class percentages:")
display((100*y_cls_test.value_counts(normalize=True).reindex([0,1], fill_value=0)).rename("percent").to_frame())
assert y_cls_train.value_counts().min() >= N_SPLITS, "Too few cases for 5-fold stratified CV."
assert TARGET_COLUMN not in X_cls_train and IDENTIFIER_COLUMN not in X_cls_train
assert len(X_cls_train) == 4000 and len(X_cls_test) == 1000

### Training-only classifier comparison

A separate stratified split preserves the risk/non-risk ratio. We compare a majority-class dummy baseline, class-weighted Logistic Regression, and class-weighted Random Forest. Five-fold stratified cross-validation uses training rows only. Model selection prioritizes at-risk recall, then F1, then precision. This favors catching more below-threshold cases but does not make false positives harmless. All reported recall/precision/F1 values use class 1 as the positive class.

In [ ]:
cls_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cls_estimators = {
    "Dummy majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                             random_state=RANDOM_STATE, n_jobs=1),
}
cls_scoring = {"precision": make_scorer(precision_score, zero_division=0),
                "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}
cls_pipelines, cls_rows = {}, []
for name, estimator in cls_estimators.items():
    pipe = make_pipeline(estimator)
    cls_pipelines[name] = pipe
    scores = cross_validate(pipe, X_cls_train, y_cls_train, cv=cls_cv,
                            scoring=cls_scoring, n_jobs=1, error_score="raise")
    cls_rows.append({"model": name,
        "CV precision mean": scores["test_precision"].mean(),
        "CV recall mean": scores["test_recall"].mean(),
        "CV recall std": scores["test_recall"].std(),
        "CV F1 mean": scores["test_f1"].mean(),
        "CV ROC-AUC mean": scores["test_roc_auc"].mean()})
cls_cv_results = pd.DataFrame(cls_rows).sort_values(
    ["CV recall mean", "CV F1 mean", "CV precision mean"], ascending=False, kind="stable")
display(cls_cv_results.round(4))
selected_classifier_name = cls_cv_results.iloc[0]["model"]
selected_classifier = cls_pipelines[selected_classifier_name]
print(f"Selected using training CV recall, then F1/precision: {selected_classifier_name}")

### Held-out classification evaluation

After selection, refit on the full classification training split and evaluate once on its held-out rows. Precision tells how many flagged examples were labelled at risk; recall tells how many labelled at-risk examples were flagged; F1 balances the two. Accuracy is context only because the majority class can dominate it. ROC-AUC is shown only when the test set has both classes and probability estimates are available. Confusion-matrix rows are actual classes; columns are predicted classes.

In [ ]:
selected_classifier.fit(X_cls_train, y_cls_train)
risk_predictions = selected_classifier.predict(X_cls_test)
risk_probabilities = (selected_classifier.predict_proba(X_cls_test)[:, 1]
                      if hasattr(selected_classifier, "predict_proba") else None)
metrics = {
    "selected model": selected_classifier_name,
    "threshold assumption": f"Final_CGPA < {RISK_THRESHOLD:.2f}",
    "precision (at risk)": precision_score(y_cls_test, risk_predictions, zero_division=0),
    "recall (at risk)": recall_score(y_cls_test, risk_predictions, zero_division=0),
    "F1 (at risk)": f1_score(y_cls_test, risk_predictions, zero_division=0),
    "accuracy (context only)": accuracy_score(y_cls_test, risk_predictions),
}
if risk_probabilities is not None and y_cls_test.nunique() == 2:
    metrics["ROC-AUC"] = roc_auc_score(y_cls_test, risk_probabilities)
else:
    metrics["ROC-AUC"] = np.nan
    print("ROC-AUC unavailable: probability scores missing or only one class in held-out labels.")
display(pd.Series(metrics, name="held-out test result").to_frame())
print(classification_report(y_cls_test, risk_predictions, labels=[0,1],
      target_names=["Not at risk", "At risk"], zero_division=0))
confusion = confusion_matrix(y_cls_test, risk_predictions, labels=[0,1])
display(pd.DataFrame(confusion, index=["Actual not at risk", "Actual at risk"],
                     columns=["Predicted not at risk", "Predicted at risk"]))
print(f"Test cases labelled at risk: {int(y_cls_test.sum())} / {len(y_cls_test)}")
print(f"Test cases predicted at risk: {int(risk_predictions.sum())} / {len(risk_predictions)}")
assert confusion.shape == (2, 2) and len(risk_predictions) == len(y_cls_test)

## 5. Interpretation and limitations

- The 2.0 cutoff is an editable project assumption. Changing it changes class balance, error costs, and evaluation scores.
- If few records are below the cutoff, recall and ROC-AUC can vary substantially across folds and splits; interpret them with the class counts.
- Results describe this dataset and split only. The dataset may be synthetic; its source and representativeness need verification before real use.
- `Previous_CGPA` is valid only if known before prediction. Predictive association does not show that changing a feature causes CGPA to change.
- Predictions are not clipped to 0 to 4. Any out-of-range outputs are reported so later decisions about bounded predictions can be evaluated transparently.
- Risk scores are decision-support estimates. False alarms can prompt unnecessary concern; missed cases can delay human follow-up. Neither is proof of a student's future outcome.

No model artifacts are saved in this module. Module 4 will analyze residuals, group performance, and explainability before deciding whether a final pipeline should be saved.